# IRWA 2026 Part 3: Ranking & Filtering

Structure:

0. Setup and data loading
1. **SCORE**
   - 1.a TF-IDF + cosine similarity
   - 1.b BM25
   - 1.c Our score
   - 1.d Comparison of the three rankings
2. **word2vec + cosine similarity** (top 20 for our 5 queries)
3. **A better representation than word2vec** (discussed in the report)
4. Extra: labels for the products the new methods bring to the top

Every method uses the **same AND search** as Part 2 (`conjunctive_search`): a product is returned only if it
contains all the query terms. The methods only change the **order** of those products.

The functions live in `myapp/search/algorithms.py` (1.a–1.c and the duplicate filter) and
`myapp/search/embeddings.py` (2).

## 0. Setup and data loading

In [1]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("../..").resolve()
sys.path.append(str(ROOT))

DATA_PATH = ROOT / "data" / "fashion_products_dataset.json"
CORPUS_PATH = ROOT / "data" / "processed_corpus.pkl"
LABELS_PATH = ROOT / "data" / "validation_labels.csv"
OUR_LABELS_PATH = ROOT / "project_progress" / "part_2" / "our_queries_labels.csv"  # Part 2 labels (top 20 of TF-IDF)
NEW_LABELS_PATH = Path("our_queries_labels_part3.csv")  # optional, see section 4

from myapp.search.preprocessing import preprocess_text, load_raw_corpus, build_processed_corpus
from myapp.search.algorithms import (
    create_index, compute_idf, compute_doc_norms, search_tfidf,                     # 1.a (Part 2)
    compute_doc_lengths, compute_bm25_idf, search_bm25,                             # 1.b
    build_product_features, our_score_components, search_our_score,                 # 1.c
    OUR_SCORE_WEIGHTS, query_gender, gender_mismatch, conjunctive_search,
    deduplicate)                                                                    # filtering
from myapp.search.evaluation import ranked_labels, evaluate_query, mean_average_precision, mean_reciprocal_rank

pd.set_option("display.max_colwidth", 50)

In [2]:
# Processed corpus from Part 1 (rebuilt from the raw JSON if the file is missing or unreadable)
try:
    corpus = pd.read_pickle(CORPUS_PATH)
except Exception:
    corpus = build_processed_corpus(load_raw_corpus(DATA_PATH))
    corpus.to_pickle(CORPUS_PATH)
corpus_by_pid = corpus.set_index("pid")

start = time.time()
N = len(corpus)
index = create_index(corpus, field="all_tokens")  # same inverted index as Part 2
idf = compute_idf(index, N)                         # 1.a
doc_norms = compute_doc_norms(index, idf)           # 1.a
bm25_idf = compute_bm25_idf(index, N)               # 1.b
doc_lengths = compute_doc_lengths(corpus)           # 1.b
features = build_product_features(corpus)           # 1.c
print(f"Index and statistics built in {time.time() - start:.1f} s")
print(f"Products: {N} | terms: {len(index)} | average length: {np.mean(list(doc_lengths.values())):.1f} terms")

Index and statistics built in 0.9 s
Products: 28080 | terms: 5790 | average length: 42.4 terms


In [3]:
VALIDATION_QUERIES = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
TEST_QUERIES = {3: "women cotton printed kurta", 4: "men black polo tshirt", 5: "women winter hooded jacket",
                6: "men running sports shoes", 7: "men formal trousers"}  # our queries from Part 2 (1.2)
ALL_QUERIES = {**VALIDATION_QUERIES, **TEST_QUERIES}

validation = pd.read_csv(LABELS_PATH, dtype={"pid": str})
ours = pd.read_csv(OUR_LABELS_PATH, dtype={"pid": str})
if NEW_LABELS_PATH.exists():  # labels added in Part 3 (section 4)
    new = pd.read_csv(NEW_LABELS_PATH, dtype={"pid": str}).dropna(subset=["labels"])
    ours = pd.concat([ours, new]).drop_duplicates(["query_id", "pid"], keep="first")
ours["labels"] = ours["labels"].astype(int)
print("Labelled products per query:", ours.groupby("query_id").size().to_dict())

Labelled products per query: {3: 20, 4: 20, 5: 20, 6: 20, 7: 20}


In [4]:
K = 10
# Every ranking method: query -> [(pid, score), ...] (all AND matches, or the top k)
RANKERS = {
    "TF-IDF": lambda q, k=None: search_tfidf(q, index, idf, doc_norms, top_k=k),
    "BM25": lambda q, k=None: search_bm25(q, index, bm25_idf, doc_lengths, top_k=k),
    "Our score": lambda q, k=None: search_our_score(q, index, bm25_idf, doc_lengths, features, top_k=k),
}
SHOW = ["title", "brand", "selling_price", "discount", "average_rating", "out_of_stock"]

def results_table(ranking):
    """Ranking [(pid, score), ...] -> table with the score and some of the output fields."""
    table = corpus_by_pid.loc[[pid for pid, _ in ranking], SHOW].reset_index()
    table.insert(0, "score", [round(score, 3) for _, score in ranking])
    table.index = range(1, len(table) + 1)
    return table

EXAMPLE = "men running sports shoes"  # query 6: the gender problem found in Part 2

## 1. SCORE

### 1.a TF-IDF + cosine similarity

The Part 2 ranking, unchanged (`search_tfidf` = `search_in_corpus`):

$$
w_{t,d} = (1 + \log tf_{t,d}) \cdot \log\frac{N}{df_t} \qquad
score(q, d) = \frac{\sum_{t \in q} w_{t,q} \, w_{t,d}}{\lVert q \rVert \, \lVert d \rVert}
$$

$\lVert d \rVert$ is the length of the **whole** product vector (all its terms, not only the query terms), so a
product with many other terms gets a lower score.

In [5]:
print(f"TF-IDF, top {K} for: {EXAMPLE}")
results_table(RANKERS["TF-IDF"](EXAMPLE, K))

TF-IDF, top 10 for: men running sports shoes


,score,pid,title,brand,selling_price,discount,average_rating,out_of_stock
1,0.690,SHOESS9ZEZ6TEQUJ,Running Shoes For Men (Multicolor),Vector,1440.0,9.0,4.3,False
2,0.690,SHOESS9ZGMH7CURM,Running Shoes For Men (Multicolor),Vector,1440.0,9.0,3.7,False
3,0.676,SHOEP5GUQYYFKYSN,Running Shoes For Women (White),Vector,990.0,9.0,4.1,False
4,0.595,SHOEWQKVSGFGXPQF,Running Shoes For Men (Blue),Vector,1520.0,1.0,3.3,False
5,0.585,SHOFTZFPNCU4VWHK,Running Shoes For Men (Black),Bone,379.0,62.0,4.1,False
6,0.579,SHOFTZFQP8W9G86F,Running Shoes For Women (Black),Bone,341.0,65.0,4.1,False
7,0.560,SHOFN3EJHXHGSGUC,RS-7250 Running Shoes For Men (Black),Vector,990.0,9.0,3.9,False
8,0.557,SHOEKNRJGEYJCDQK,"SPRINT Running Shoes For Men (Blue, Grey)",Vector,490.0,1.0,3.7,False
9,0.556,SHOFTZFQQYKC8HY6,Running Shoes For Men (Yellow),Bone,341.0,65.0,4.1,False
10,0.556,SHOEKNRJ34UCMAPA,"SPRINT Running Shoes For Men (Black, Grey)",Vector,490.0,1.0,3.7,False


### 1.b BM25

$$
score(q, d) = \sum_{t \in q} idf_{BM25}(t) \cdot \frac{tf_{t,d} \, (k_1 + 1)}{tf_{t,d} + k_1 \left(1 - b + b \, \frac{|d|}{avg|d|}\right)}
\qquad idf_{BM25}(t) = \log\left(1 + \frac{N - df_t + 0.5}{df_t + 0.5}\right)
$$

* $k_1 = 1.2$: **tf saturation**. A second occurrence of a term helps, a tenth one almost not.
* $b = 0.75$: **length normalisation** relative to the average product length (42.4 terms). Only the
  length counts, not which other terms the product has.
* The idf has a $+1$ inside the log so it is never negative: the classic BM25 idf, $\log\frac{N - df + 0.5}{df + 0.5}$,
  is negative for terms in more than half of the products (`cotton`, `sleev`, `men`; `women` is just above 0), so a product would
  *lose* points for matching them. With the $+1$ it is almost the same as $\log(N/df)$ (table below): the
  difference between TF-IDF and BM25 comes from **tf saturation and length normalisation**, not from the idf.
* $k_1 = 1.2$ and $b = 0.75$ are the usual default values (e.g. in Lucene/Elasticsearch); we did not tune them
  on our few labels.

In [6]:
# idf of the query terms: TF-IDF, classic BM25 (without +1) and the BM25 idf we use (with +1)
terms = sorted({t for q in ALL_QUERIES.values() for t in preprocess_text(q)}, key=lambda t: -len(index[t]))
pd.DataFrame({"products (df)": [len(index[t]) for t in terms],
              "idf TF-IDF": [round(idf[t], 3) for t in terms],
              "idf BM25 classic": [round(np.log((N - len(index[t]) + 0.5) / (len(index[t]) + 0.5)), 3) for t in terms],
              "idf BM25 (ours, +1)": [round(bm25_idf[t], 3) for t in terms]}, index=terms)

,products (df),idf TF-IDF,idf BM25 classic,"idf BM25 (ours, +1)"
cotton,19584,0.360,-0.835,0.360
sleev,17904,0.450,-0.565,0.450
men,14123,0.687,-0.012,0.687
women,13699,0.718,0.049,0.718
tshirt,12899,0.778,0.163,0.778
print,9448,1.089,0.679,1.089
blue,7154,1.367,1.073,1.367
full,7150,1.368,1.074,1.368
black,6163,1.516,1.269,1.516
slim,5646,1.604,1.380,1.604


In [7]:
print(f"BM25, top {K} for: {EXAMPLE}")
results_table(RANKERS["BM25"](EXAMPLE, K))

BM25, top 10 for: men running sports shoes


,score,pid,title,brand,selling_price,discount,average_rating,out_of_stock
1,17.541,SHOFN3EJHXHGSGUC,RS-7250 Running Shoes For Men (Black),Vector,990.0,9.0,3.9,False
2,17.541,SHOFN3EJVGC9VXYT,RS-7200 Running Shoes For Men (Navy),Vector,990.0,9.0,3.9,False
3,17.261,SHOFN3EFNYGBKZT7,"RS-7050 Running Shoes For Men (Blue, Navy)",Vector,810.0,9.0,3.9,False
4,17.208,SHOFN3EJZVGC6GH8,RS-7250 Running Shoes For Women (Navy),Vector,990.0,9.0,3.9,False
5,17.076,SHOESS9ZEZ6TEQUJ,Running Shoes For Men (Multicolor),Vector,1440.0,9.0,4.3,False
6,17.076,SHOESS9ZGMH7CURM,Running Shoes For Men (Multicolor),Vector,1440.0,9.0,3.7,False
7,16.711,SHOEWQKVSGFGXPQF,Running Shoes For Men (Blue),Vector,1520.0,1.0,3.3,False
8,16.583,SHOEP5GUQYYFKYSN,Running Shoes For Women (White),Vector,990.0,9.0,4.1,False
9,16.535,SHOEKNRJ34UCMAPA,"SPRINT Running Shoes For Men (Black, Grey)",Vector,490.0,1.0,3.7,False
10,16.535,SHOEKNRJGEYJCDQK,"SPRINT Running Shoes For Men (Blue, Grey)",Vector,490.0,1.0,3.7,False


### 1.c Our score

Our score starts from BM25 (text relevance) and adds what Part 2 showed was missing: the **title**, the
**numeric fields** (rating, discount), the **stock** and the **gender**.

$$
score(q, d) = \Big(\underbrace{\tfrac{BM25(q,d)}{\max_{d'} BM25(q,d')}}_{text} + 0.3 \cdot title + 0.1 \cdot rating + 0.05 \cdot discount\Big)
\times 0.5^{[\text{out of stock}]} \times 0.5^{[\text{wrong gender}]}
$$

| Part | Value | Why |
|---|---|---|
| text | BM25 divided by the best BM25 of the query, in (0, 1] | relevance stays the main factor; dividing makes it comparable with the other parts |
| title | share of the query terms that appear in the **title** | a query word in the title says what the product *is* (Part 1: title 6.5 words vs description 49) |
| rating | (rating − 3) / 2, in [−1, 1]; **0 if no rating** | good products up, bad ones down; an unrated product is neutral, we don't invent a rating (Part 1, `has_rating`) |
| discount | discount / 100 | small bonus; mostly decides between identical products (42% duplicates) |
| × 0.5 out of stock | | the user can't buy it: pushed down, not deleted (Part 2: out-of-stock products at ranks 1 and 5) |
| × 0.5 wrong gender | query says men/women and the product's `Ideal For` or title says the other one | Part 2: 8 of the top 20 for "men running sports shoes" were women's shoes |

Price is **not** in the score: cheaper is not more relevant, it is a user preference (better as a filter).

In [8]:
pd.Series(OUR_SCORE_WEIGHTS, name="weight").to_frame()

,weight
title,0.30
rating,0.10
discount,0.05
out_of_stock,0.50
gender_mismatch,0.50


In [9]:
# Every part of our score for the top 10 of the example query
rows = our_score_components(preprocess_text(EXAMPLE), conjunctive_search(preprocess_text(EXAMPLE), index),
                            index, bm25_idf, doc_lengths, features)[:K]
parts = pd.DataFrame(rows).set_index("pid").round(3)
parts.insert(0, "title text", corpus_by_pid.loc[parts.index, "title"].str[:40])
parts.reset_index(drop=True).set_axis(range(1, len(parts) + 1))

,title text,score,text,title,rating,discount,out_of_stock,gender_mismatch
1,RS-7250 Running Shoes For Men (Black),1.274,1.000,0.75,0.45,0.09,False,False
2,RS-7200 Running Shoes For Men (Navy),1.274,1.000,0.75,0.45,0.09,False,False
3,Running Shoes For Men (Multicolor),1.268,0.973,0.75,0.65,0.09,False,False
4,"RS-7050 Running Shoes For Men (Blue, Na",1.258,0.984,0.75,0.45,0.09,False,False
5,Running Shoes For Men (Yellow),1.255,0.943,0.75,0.55,0.65,False,False
6,Running Shoes For Men (Black),1.254,0.943,0.75,0.55,0.62,False,False
7,Running Shoes For Men (Multicolor),1.238,0.973,0.75,0.35,0.09,False,False
8,"SPRINT Running Shoes For Men (Black, Gr",1.203,0.943,0.75,0.35,0.01,False,False
9,"SPRINT Running Shoes For Men (Blue, Gre",1.203,0.943,0.75,0.35,0.01,False,False
10,Running Shoes For Men (Blue),1.193,0.953,0.75,0.15,0.01,False,False


In [10]:
# How many matching products each penalty affects, for every query
affected = []
for qid, q in ALL_QUERIES.items():
    terms = preprocess_text(q)
    rows = our_score_components(terms, conjunctive_search(terms, index), index, bm25_idf, doc_lengths, features)
    affected.append({"query": qid, "AND results": len(rows),
                    "out of stock": sum(r["out_of_stock"] for r in rows),
                    "wrong gender": sum(r["gender_mismatch"] for r in rows),
                    "no rating": sum(features[r["pid"]]["rating"] is None for r in rows)})
pd.DataFrame(affected).set_index("query")

,AND results,out of stock,wrong gender,no rating
query,,,,
1,467,38,5,18
2,219,34,0,28
3,41,3,0,6
4,222,7,0,9
5,72,7,0,1
6,74,3,39,2
7,237,11,0,5


### 1.d Comparison of the three rankings

**Side by side.** The top 10 of each method for every query (`[OOS]` = out of stock, `[W]` = wrong gender).

In [11]:
def label_title(pid, q_gender):
    """Short title + markers for out of stock [OOS] and wrong gender [W]."""
    text = corpus_by_pid.at[pid, "title"].replace("\xa0", " ")[:45]
    if features[pid]["out_of_stock"]:
        text += " [OOS]"
    if gender_mismatch(q_gender, features[pid]["gender"]):
        text += " [W]"
    return text

def side_by_side(query, k=K):
    q_gender = query_gender(preprocess_text(query))
    return pd.DataFrame({name: [label_title(pid, q_gender) for pid, _ in ranker(query, k)]
                         for name, ranker in RANKERS.items()}, index=range(1, k + 1))

for qid, q in ALL_QUERIES.items():
    print(f"Query {qid}: {q}")
    display(side_by_side(q))

Query 1: women full sleeve sweatshirt cotton


,TF-IDF,BM25,Our score
1,Full Sleeve Solid Women Sweatshirt [OOS],Full Sleeve Printed Women Sweatshirt,Full Sleeve Solid Women Sweatshirt
2,Full Sleeve Printed Women Sweatshirt,Full Sleeve Solid Women Sweatshirt [OOS],Full Sleeve Solid Women Sweatshirt
3,Full Sleeve Printed Women Sweatshirt,Full Sleeve Striped Women Sweatshirt,Full Sleeve Printed Women Sweatshirt
4,Full Sleeve Printed Women Sweatshirt,Full Sleeve Striped Women Sweatshirt,Full Sleeve Solid Women Sweatshirt
5,Full Sleeve Printed Women Sweatshirt [OOS],Full Sleeve Solid Women Sweatshirt,Full Sleeve Solid Women Sweatshirt
6,Full Sleeve Printed Women Sweatshirt,Full Sleeve Solid Women Sweatshirt,Full Sleeve Striped Women Sweatshirt
7,Full Sleeve Printed Women Sweatshirt,Full Sleeve Solid Women Sweatshirt,Full Sleeve Solid Women Sweatshirt
8,Full Sleeve Solid Women Sweatshirt,Full Sleeve Solid Women Sweatshirt,Full Sleeve Solid Women Sweatshirt
9,Full Sleeve Solid Women Sweatshirt,Full Sleeve Printed Women Sweatshirt,Full Sleeve Embroidered Women Sweatshirt
10,Full Sleeve Solid Women Sweatshirt,Full Sleeve Embroidered Women Sweatshirt,Full Sleeve Striped Women Sweatshirt


Query 2: men slim jeans blue


,TF-IDF,BM25,Our score
1,Slim Men Blue Jeans [OOS],Slim Men Dark Blue Jeans,Slim Men Dark Blue Jeans
2,Slim Men Blue Jeans [OOS],Slim Men Blue Jeans,Slim Men Blue Jeans
3,Slim Men Blue Jeans,Slim Men Dark Blue Jeans,Slim Men Blue Jeans
4,Slim Men Blue Jeans,Slim Men Blue Jeans,Slim Men Blue Jeans
5,Slim Men Blue Jeans,Slim Men Dark Blue Jeans,Slim Men Dark Blue Jeans
6,Slim Men Blue Jeans,Slim Men Blue Jeans,Slim Men Dark Blue Jeans
7,Slim Men Blue Jeans,Slim Men Blue Jeans,Slim Men Blue Jeans
8,Slim Men Blue Jeans,Slim Men Blue Jeans,Slim Men Blue Jeans
9,Slim Men Blue Jeans,Slim Men Blue Jeans,Slim Men Blue Jeans
10,Slim Men Blue Jeans,Slim Men Blue Jeans [OOS],Slim Men Blue Jeans


Query 3: women cotton printed kurta


,TF-IDF,BM25,Our score
1,Cotton Printed Kurta Fabric (Unstitched),Women Printed Pure Cotton Straight Kurta (Br,Women Printed Pure Cotton Straight Kurta (Br
2,Women Kurta and Pyjama Set Cotton Blend,Women Printed Cotton Silk Straight Kurta (Be,Women Printed Cotton Linen Blend Straight Kur
3,Women Kurta and Pyjama Set Cotton Blend,Cotton Printed Kurta Fabric (Unstitched),Women Printed Cotton Linen Blend Straight Kur
4,Women Kurta and Pyjama Set Cotton Blend,Women Printed Cotton Linen Blend Straight Kur,Women Printed Cotton Linen Blend Straight Kur
5,Women Printed Cotton Linen Blend Straight Kur,Women Printed Cotton Linen Blend Straight Kur,Women Printed Cotton Linen Blend Straight Kur
6,Women Printed Cotton Linen Blend Straight Kur,Women Printed Cotton Linen Blend Straight Kur,Women Printed Cotton Linen Blend Straight Kur
7,Women Printed Cotton Linen Blend Straight Kur,Women Printed Cotton Linen Blend Straight Kur,Women Printed Pure Cotton Straight Kurta (Da
8,Women Printed Cotton Silk Straight Kurta (Be,Women Printed Cotton Linen Blend Straight Kur,Women Printed Pure Cotton Straight Kurta (Li
9,Women Kurta and Pyjama Set Pure Cotton,Women Printed Cotton Blend Straight Kurta (O,Women Printed Cotton Silk Straight Kurta (Be
10,Women Kurta and Pyjama Set Pure Cotton,Women Printed Pure Cotton Straight Kurta (Da,Women Printed Cotton Blend Straight Kurta (O


... (4 outputs left out)

Query 6: men running sports shoes


,TF-IDF,BM25,Our score
1,Running Shoes For Men (Multicolor),RS-7250 Running Shoes For Men (Black),RS-7250 Running Shoes For Men (Black)
2,Running Shoes For Men (Multicolor),RS-7200 Running Shoes For Men (Navy),RS-7200 Running Shoes For Men (Navy)
3,Running Shoes For Women (White) [W],"RS-7050 Running Shoes For Men (Blue, Navy)",Running Shoes For Men (Multicolor)
4,Running Shoes For Men (Blue),RS-7250 Running Shoes For Women (Navy) [W],"RS-7050 Running Shoes For Men (Blue, Navy)"
5,Running Shoes For Men (Black),Running Shoes For Men (Multicolor),Running Shoes For Men (Yellow)
6,Running Shoes For Women (Black) [W],Running Shoes For Men (Multicolor),Running Shoes For Men (Black)
7,RS-7250 Running Shoes For Men (Black),Running Shoes For Men (Blue),Running Shoes For Men (Multicolor)
8,"SPRINT Running Shoes For Men (Blue, Grey)",Running Shoes For Women (White) [W],"SPRINT Running Shoes For Men (Black, Grey)"
9,Running Shoes For Men (Yellow),"SPRINT Running Shoes For Men (Black, Grey)","SPRINT Running Shoes For Men (Blue, Grey)"
10,"SPRINT Running Shoes For Men (Black, Grey)","SPRINT Running Shoes For Men (Blue, Grey)",Running Shoes For Men (Blue)


Query 7: men formal trousers


,TF-IDF,BM25,Our score
1,Slim Fit Men White Pure Cotton Trousers,Regular Fit Men Black Viscose Trousers,Slim Fit Men Black Polyester Viscose Blend Tr
2,Slim Fit Men Blue Cotton Blend Trousers,Slim Fit Men Beige Viscose Blend Trousers,Slim Fit Men Beige Viscose Blend Trousers
3,Slim Fit Men Blue Cotton Blend Trousers,Slim Fit Men Gold Viscose Blend Trousers,Slim Fit Men Gold Viscose Blend Trousers
4,Slim Fit Men Dark Blue Pure Cotton Trousers,Slim Fit Men Grey Viscose Blend Trousers,Slim Fit Men Grey Viscose Blend Trousers
5,Slim Fit Men Blue Cotton Blend Trousers,Slim Fit Men Black Viscose Blend Trousers,Slim Fit Men Black Viscose Blend Trousers
6,Slim Fit Men Blue Cotton Blend Trousers,Regular Fit Men Black Viscose Blend Trousers,Slim Fit Men Blue Polyester Viscose Blend Tro
7,Regular Fit Men Black Cotton Blend Trousers,Slim Fit Men Dark Blue Pure Cotton Trousers,Slim Fit Men Grey Rayon Trousers
8,Slim Fit Men Green Cotton Blend Trousers,Slim Fit Men Grey Rayon Trousers,Regular Fit Men Black Polyester Viscose Blend
9,Slim Fit Men Black Polyester Trousers,Regular Fit Men Black Cotton Blend Trousers,Slim Fit Men Green Cotton Blend Trousers
10,Slim Fit Men Beige Cotton Blend Trousers,Regular Fit Men Black Polyester Viscose Blend,Slim Fit Men Blue Cotton Viscose Blend Trouse


**Quality of the top 10.** For each method and query: distinct titles (duplicates), out-of-stock products,
average rating, wrong-gender products, average length of the products (terms) and how many products are
also in the TF-IDF top 10.

In [12]:
def top_k_quality(ranker, queries, k=K):
    rows = []
    for qid, q in queries.items():
        top = [pid for pid, _ in ranker(q, k)]
        q_gender = query_gender(preprocess_text(q))
        products = corpus_by_pid.loc[top]
        tfidf_top = {pid for pid, _ in RANKERS["TF-IDF"](q, k)}
        rows.append({"query": qid,
                     "distinct titles": products["title"].nunique(),
                     "out of stock": int(products["out_of_stock"].sum()),
                     "avg rating": round(products["average_rating"].mean(), 2),
                     "wrong gender": sum(gender_mismatch(q_gender, features[p]["gender"]) for p in top),
                     "avg length": round(np.mean([doc_lengths[p] for p in top]), 1),
                     "same as TF-IDF": len(set(top) & tfidf_top)})
    return pd.DataFrame(rows).set_index("query")

quality = {name: top_k_quality(ranker, ALL_QUERIES) for name, ranker in RANKERS.items()}
for name, table in quality.items():
    print(name)
    display(table)

TF-IDF


,distinct titles,out of stock,avg rating,wrong gender,avg length,same as TF-IDF
query,,,,,,
1,2,2,3.41,0,22.0,10
2,1,2,3.68,0,18.4,10
3,6,0,3.29,0,26.4,10
4,2,1,3.82,0,25.9,10
5,3,1,3.97,0,30.1,10
6,9,0,3.90,2,14.4,10
7,7,0,3.30,0,25.4,10


BM25


,distinct titles,out of stock,avg rating,wrong gender,avg length,same as TF-IDF
query,,,,,,
1,4,1,3.97,0,31.9,2
2,2,1,3.45,0,27.4,0
3,8,0,3.78,0,34.0,5
4,7,0,3.08,0,32.2,0
5,5,4,3.78,0,44.3,3
6,9,0,3.84,2,15.5,7
7,10,0,3.45,0,24.2,2


Our score


,distinct titles,out of stock,avg rating,wrong gender,avg length,same as TF-IDF
query,,,,,,
1,4,0,4.05,0,31.3,2
2,2,0,3.98,0,24.5,3
3,8,0,4.02,0,36.6,4
4,2,0,4.22,0,31.4,1
5,4,0,4.21,0,37.2,5
6,9,0,3.86,0,15.4,8
7,10,0,3.98,0,24.7,1


In [13]:
# Averages over the 7 queries (totals for out of stock and wrong gender)
summary = pd.DataFrame({name: {"distinct titles (avg)": t["distinct titles"].mean(),
                               "out of stock (total)": t["out of stock"].sum(),
                               "avg rating": t["avg rating"].mean(),
                               "wrong gender (total)": t["wrong gender"].sum(),
                               "avg length (terms)": t["avg length"].mean(),
                               "same as TF-IDF (avg)": t["same as TF-IDF"].mean()}
                        for name, t in quality.items()}).round(2)
summary

,TF-IDF,BM25,Our score
distinct titles (avg),4.29,6.43,5.57
out of stock (total),6.00,6.00,0.00
avg rating,3.62,3.62,4.05
wrong gender (total),2.00,2.00,0.00
avg length (terms),23.23,29.93,28.73
same as TF-IDF (avg),10.00,2.71,3.43


**Evaluation metrics** (Part 2 functions, K = 10). As in Part 2, each ranking keeps only the labelled products,
in its own order. Our labels only cover the **top 20 of TF-IDF**, so BM25 and our score can bring unlabelled
products into their top 10. Those are skipped, which makes the comparison less reliable. The column
`judged in top 10` shows how many of the first 10 products have a label. The validation labels are far down the
ranking for every method (no labelled product in any top 10 of query 1, at most one for query 2), so they say little
about the top 10.

In [14]:
def evaluate(ranker, queries, labels_df, k=K):
    """Every metric for each query, plus MAP and MRR (rounded to 3 decimals)."""
    rows, per_query = {}, []
    for qid, q in queries.items():
        truth = labels_df[labels_df["query_id"] == qid]
        labels = dict(zip(truth["pid"], truth["labels"]))
        total_relevant = int(truth["labels"].sum())
        ranking = [pid for pid, _ in ranker(q)]
        rels = ranked_labels(ranking, labels)
        rows[f"Query {qid}"] = {**evaluate_query(rels, k, total_relevant),
                                "judged in top 10": sum(pid in labels for pid in ranking[:k])}
        per_query.append((rels, total_relevant))
    summary = {"MAP": mean_average_precision(per_query, k), "MRR": mean_reciprocal_rank([r for r, _ in per_query], k)}
    return pd.DataFrame(rows).T.round(3), summary

metrics = []
for name, ranker in RANKERS.items():
    val_table, val_summary = evaluate(ranker, VALIDATION_QUERIES, validation)
    our_table, our_summary = evaluate(ranker, TEST_QUERIES, ours)
    print(name)
    display(pd.concat([val_table, our_table]))
    metrics.append({"method": name, "MAP validation": val_summary["MAP"], "MRR validation": val_summary["MRR"],
                    "MAP our queries": our_summary["MAP"], "MRR our queries": our_summary["MRR"],
                    "judged in top 10 (our queries, avg)": our_table["judged in top 10"].mean()})
pd.DataFrame(metrics).set_index("method").round(3)

TF-IDF


,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10,judged in top 10
Query 1,1.0,0.769,1.000,0.870,1.0,1.000,0.0
Query 2,0.6,0.600,0.569,0.600,1.0,0.716,1.0
Query 3,0.9,0.500,0.707,0.643,0.5,0.780,10.0
Query 4,1.0,0.500,1.000,0.667,1.0,1.000,10.0
Query 5,0.7,0.500,0.700,0.583,1.0,0.801,10.0
Query 6,0.8,0.667,0.659,0.727,1.0,0.812,10.0
Query 7,1.0,0.500,1.000,0.667,1.0,1.000,10.0


BM25


,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10,judged in top 10
Query 1,1.0,0.769,1.000,0.870,1.0,1.000,0.0
Query 2,0.6,0.600,0.586,0.600,1.0,0.722,1.0
Query 3,0.9,0.500,0.790,0.643,1.0,0.890,10.0
Query 4,1.0,0.500,1.000,0.667,1.0,1.000,1.0
Query 5,1.0,0.714,1.000,0.833,1.0,1.000,9.0
Query 6,0.8,0.667,0.707,0.727,1.0,0.836,10.0
Query 7,1.0,0.500,1.000,0.667,1.0,1.000,4.0


Our score


,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10,judged in top 10
Query 1,0.9,0.692,0.890,0.783,1.0,0.934,0.0
Query 2,0.6,0.600,0.586,0.600,1.0,0.722,0.0
Query 3,1.0,0.556,1.000,0.714,1.0,1.000,9.0
Query 4,1.0,0.500,1.000,0.667,1.0,1.000,4.0
Query 5,1.0,0.714,1.000,0.833,1.0,1.000,9.0
Query 6,1.0,0.833,1.000,0.909,1.0,1.000,10.0
Query 7,1.0,0.500,1.000,0.667,1.0,1.000,4.0


,MAP validation,MRR validation,MAP our queries,MRR our queries,"judged in top 10 (our queries, avg)"
method,,,,,
TF-IDF,0.785,1.0,0.813,0.9,10.0
BM25,0.793,1.0,0.899,1.0,6.8
Our score,0.738,1.0,1.000,1.0,7.2


**Why TF-IDF and BM25 differ: document length.** Cosine divides by the length of the whole product vector, so
short products win. BM25 only compares the product length with the average, softened by $b$. The table shows
the average length of the top 10 for TF-IDF and for BM25 with different $b$ ($b = 0$: length ignored).

In [15]:
rows = {"TF-IDF": [np.mean([doc_lengths[p] for p, _ in RANKERS["TF-IDF"](q, K)]) for q in ALL_QUERIES.values()]}
for b in [0.0, 0.5, 0.75, 1.0]:
    rows[f"BM25 b={b}"] = [np.mean([doc_lengths[p] for p, _ in search_bm25(q, index, bm25_idf, doc_lengths, top_k=K, b=b)])
                          for q in ALL_QUERIES.values()]
pd.DataFrame(rows, index=[f"Query {qid}" for qid in ALL_QUERIES]).round(1).T.assign(average=lambda t: t.mean(axis=1).round(1))

,Query 1,Query 2,Query 3,Query 4,Query 5,Query 6,Query 7,average
TF-IDF,22.0,18.4,26.4,25.9,30.1,14.4,25.4,23.2
BM25 b=0.0,70.6,82.5,90.0,178.6,55.6,79.3,75.7,90.3
BM25 b=0.5,31.9,30.5,39.6,32.8,47.2,19.7,51.4,36.2
BM25 b=0.75,31.9,27.4,34.0,32.2,44.3,15.5,24.2,29.9
BM25 b=1.0,24.9,19.4,27.9,30.2,35.7,15.5,23.0,25.2


**Filtering duplicates.** Our score already breaks the ties between identical products (rating, stock and
discount differ), but the copies are still there. `deduplicate` keeps only the best-ranked product of each group
with the same title, brand and description.

In [16]:
rows = []
for qid, q in ALL_QUERIES.items():
    full = RANKERS["Our score"](q)
    unique = deduplicate(full, corpus_by_pid)
    top_before = corpus_by_pid.loc[[p for p, _ in full[:K]], "title"].nunique()
    top_after = corpus_by_pid.loc[[p for p, _ in unique[:K]], "title"].nunique()
    rows.append({"query": qid, "AND results": len(full), "after removing duplicates": len(unique),
                 "distinct titles in top 10 (before)": top_before, "distinct titles in top 10 (after)": top_after})
pd.DataFrame(rows).set_index("query")

,AND results,after removing duplicates,distinct titles in top 10 (before),distinct titles in top 10 (after)
query,,,,
1,467,281,4,4
2,219,153,2,2
3,41,28,8,10
4,222,180,2,2
5,72,42,4,6
6,74,70,9,10
7,237,156,10,10


In [17]:
q = "men slim jeans blue"
print("Our score + duplicate filter, top 10 for:", q)
results_table(deduplicate(RANKERS["Our score"](q), corpus_by_pid)[:K])

Our score + duplicate filter, top 10 for: men slim jeans blue


,score,pid,title,brand,selling_price,discount,average_rating,out_of_stock
1,1.369,JEAF2QF6HNZHDXVX,Slim Men Dark Blue Jeans,Absolu,856.0,57.0,3.8,False
2,1.333,JEAFESND4QWQUBZD,Slim Men Blue Jeans,True Bl,1319.0,60.0,5.0,False
3,1.327,JEAF8HYN5B9ZU6FP,Slim Men Blue Jeans,Absolu,549.0,64.0,3.8,False
4,1.305,JEAF8HYNUMGZMQU5,Slim Men Blue Jeans,Absolu,549.0,64.0,3.8,False
5,1.305,JEAFJM2P2FQQJF3C,Slim Men Dark Blue Jeans,Absolu,589.0,61.0,4.1,False
6,1.300,JEAEVJGUSXRNSHRY,Slim Men Dark Blue Jeans,Absolu,489.0,53.0,3.8,False
7,1.264,JEAFS2JZQBHZSHAV,Slim Men Blue Jeans,CANTAB,1499.0,50.0,3.5,False
8,1.259,JEAFSKYHUXWYKBZK,Slim Men Blue Jeans,ECKO Unl,974.0,35.0,4.1,False
9,1.254,JEAFS57EYDGY4EJM,Slim Men Blue Jeans,CANTAB,1499.0,50.0,3.3,False
10,1.254,JEAFS2JZF6ZS6X3G,Slim Men Blue Jeans,CANTAB,1399.0,50.0,3.3,False


## 2. word2vec + cosine similarity

We train word2vec (gensim) on our own processed products, so the vectors use the same stems as the index and
the queries (`sleev`, `jean`). Settings: skip-gram, 100 dimensions, window 5, words seen at least twice, 20
epochs, each distinct token list used once (42% of the products are copies), one worker and a fixed seed so the
vectors are the same in every run.

A product (or a query) is represented by the **average of its word vectors**, and the products returned by the
AND search are ranked by the cosine similarity between the query vector and the product vector.

Install gensim once: `pip install gensim==4.4.0` (in `requirements.txt`).

In [18]:
from myapp.search.embeddings import train_word2vec, compute_doc_vectors, search_word2vec, text_vector, cosine

start = time.time()
w2v = train_word2vec(corpus)
print(f"word2vec trained in {time.time() - start:.1f} s | vocabulary: {len(w2v.wv.key_to_index)} words "
      f"| vector size: {w2v.wv.vector_size}")
start = time.time()
doc_vectors = compute_doc_vectors(corpus, w2v)
print(f"Product vectors computed in {time.time() - start:.1f} s")

word2vec trained in 35.4 s | vocabulary: 4379 words | vector size: 100
Product vectors computed in 1.2 s


In [19]:
# What the vectors learned: most similar words to some query terms
pd.DataFrame({term: [f"{w} ({s:.2f})" for w, s in w2v.wv.most_similar(term, topn=8)]
              for term in ["jean", "kurta", "jacket", "shoe", "black", "cotton", "men", "women"]})

,jean,kurta,jacket,shoe,black,cotton,men,women
0,skinni (0.61),ethnic (0.74),colorblock (0.63),kabaddi (0.62),grey (0.90),sammi (0.56),women (0.97),men (0.97)
1,denim (0.52),churidar (0.61),quilt (0.62),footwear (0.62),blue (0.83),trailor (0.53),white (0.61),white (0.61)
2,rise (0.48),neru (0.60),puffer (0.57),razor20 (0.62),white (0.83),elasten (0.53),alderman (0.61),blue (0.60)
3,chino (0.48),mah (0.58),bomber (0.55),lace (0.58),maroon (0.77),blend (0.52),navyorang (0.60),black (0.60)
4,knockout (0.48),satdevangi (0.58),regulor (0.54),fizer (0.56),brown (0.75),frank (0.52),black (0.60),maroon (0.60)
5,brandonloop (0.47),pathani (0.57),bridg (0.53),rs5050 (0.56),yellow (0.72),alderman (0.52),blue (0.60),yellow (0.59)
6,firstlau (0.47),tussar (0.55),nehru (0.53),bb19 (0.56),red (0.72),trouserbeig (0.51),redemb (0.60),grey (0.59)
7,attain (0.46),dupion (0.53),gap (0.51),chaser (0.56),dark (0.72),punto (0.51),yellow (0.60),130710 (0.57)


In [20]:
# Are men and women opposite? For word2vec they are very similar (they appear in the same contexts)
for a, b in [("men", "women"), ("jean", "trouser"), ("sweatshirt", "jacket"), ("black", "white")]:
    print(f"cosine({a}, {b}) = {cosine(w2v.wv[a], w2v.wv[b]):.3f}")

cosine(men, women) = 0.974
cosine(jean, trouser) = 0.450
cosine(sweatshirt, jacket) = 0.314
cosine(black, white) = 0.831


In [21]:
RANKERS["word2vec"] = lambda q, k=None: search_word2vec(q, index, w2v, doc_vectors, top_k=k)

for qid, q in TEST_QUERIES.items():
    print(f"Query {qid}: {q} (word2vec + cosine, top 20)")
    display(results_table(RANKERS["word2vec"](q, 20)))

Query 3: women cotton printed kurta (word2vec + cosine, top 20)


,score,pid,title,brand,selling_price,discount,average_rating,out_of_stock
1,0.870,KTAFWKJYFQBAAAQC,Women Polka Print Cotton Blend A-line Kurta (...,FOREVER YOU,629.0,62.0,4.6,False
2,0.863,KTAF7YABEAXUHWGF,Women Printed Cotton Silk Straight Kurta (Beige),Purple Sta,719.0,72.0,3.3,False
3,0.856,KTAFTQVJBV5TCHZP,Women Printed Pure Cotton Straight Kurta (White),allan pet,629.0,64.0,4.1,False
4,0.852,KTAFZCH5BCNAFZJ6,Women Printed Cotton Linen Blend Straight Kurt...,Mah,519.0,74.0,4.1,False
5,0.850,KTAFWKJYBGSFZHHR,Women Polka Print Cotton Blend A-line Kurta (...,FOREVER YOU,629.0,62.0,4.6,False
6,0.847,KTAFZCH5954HZSWJ,Women Printed Cotton Linen Blend Straight Kurt...,Mah,519.0,74.0,4.1,False
7,0.847,KTAFZCK5RGNECHHH,Women Printed Cotton Linen Blend Straight Kurt...,Mah,519.0,74.0,4.1,False
8,0.845,KTAFZCK5DPNRZXHZ,Women Printed Cotton Linen Blend Straight Kurt...,Mah,519.0,74.0,4.1,False
9,0.844,KTAFZCK5XEZCZXF8,Women Printed Cotton Linen Blend Straight Kurt...,Mah,519.0,74.0,4.1,False
10,0.843,KTAF5SD3WHM2V9HZ,Women Printed Cotton Blend Straight Kurta (Or...,LDHSA,599.0,33.0,4.1,False


Query 4: men black polo tshirt (word2vec + cosine, top 20)


,score,pid,title,brand,selling_price,discount,average_rating,out_of_stock
1,0.881,TSHFME2FEYDYCXZX,Striped Men Polo Neck Black T-Shirt,Steenb,699.0,36.0,3.7,False
2,0.879,TSHFJRJFWWQNCR9J,Solid Men Polo Neck Black T-Shirt,ARBO,421.0,41.0,4.2,False
3,0.877,TSHFW8H9KKGQGWJS,"Solid Men Polo Neck Black, Orange T-Shirt (Pa...",prani,582.0,65.0,3.9,False
4,0.875,TSHFUGNWKWGPZ32S,Solid Men Polo Neck Black T-Shirt,U.S.Polo As,789.0,43.0,4.3,False
5,0.874,TSHFJRHKZK89DWNR,Solid Men Polo Neck Black T-Shirt,ARBO,421.0,41.0,4.2,False
6,0.873,TSHFYEA79YGJG8UX,Solid Men Polo Neck Black T-Shirt,Crocks Cl,599.0,60.0,3.0,False
7,0.873,TSHFXMHDJYV7ZPJC,Typography Men Polo Neck Black T-Shirt,ATTIITU,599.0,60.0,2.3,False
8,0.872,TSHFYMXPFSYGNYKB,Solid Men Polo Neck Black T-Shirt,Alderman Sheikhj,429.0,52.0,NaN,False
9,0.871,TSHFJYA6GFTKKKSZ,"Striped Men Polo Neck Red, Black T-Shirt",ARBO,379.0,49.0,3.9,False
10,0.867,TSHFQEF7FAMHXGSZ,Printed Men Polo Neck Black T-Shirt,Lev,1079.0,50.0,4.6,True


Query 5: women winter hooded jacket (word2vec + cosine, top 20)


,score,pid,title,brand,selling_price,discount,average_rating,out_of_stock
1,0.808,JCKFWYTHH3B5GKWS,Full Sleeve Colorblock Women Casual Jacket,JACK AND HAR,699.0,76.0,4.1,False
2,0.807,JCKFQGP9SNRSZAKZ,Full Sleeve Colorblock Women Sports Jacket,M7 By Metrona,1180.0,60.0,4.6,False
3,0.798,JCKFW7ERKBA4BHY3,Full Sleeve Solid Women Casual Jacket,JACK AND HAR,499.0,75.0,3.9,False
4,0.796,JCKFVY254GHHUBX4,Full Sleeve Solid Women Padded Jacket,Szto,2499.0,50.0,3.8,True
5,0.795,JCKFU9APAQJBN23B,Full Sleeve Solid Women Casual Jacket,JACK AND HAR,699.0,76.0,4.3,False
6,0.795,JCKFVVHNFQYBYNGH,Full Sleeve Striped Women Casual Jacket,JACK AND HAR,699.0,76.0,3.0,True
7,0.791,JCKFVVHNRGWDGNQG,Full Sleeve Striped Women Casual Jacket,JACK AND HAR,699.0,76.0,3.0,True
8,0.789,JCKFWYNCWEHNZCTJ,Full Sleeve Striped Women Casual Jacket,JACK AND HAR,699.0,76.0,3.4,True
9,0.788,JCKFVEVFZ6BGWHFZ,Full Sleeve Solid Women Sports Jacket,REEB,1799.0,35.0,4.0,False
10,0.787,JCKFPMGU43VZYBGC,Full Sleeve Solid Women Casual Jacket,Pu,4799.0,40.0,4.3,False


Query 6: men running sports shoes (word2vec + cosine, top 20)


,score,pid,title,brand,selling_price,discount,average_rating,out_of_stock
1,0.861,SHOFN3EJHXHGSGUC,RS-7250 Running Shoes For Men (Black),Vector,990.0,9.0,3.9,False
2,0.859,SHOEKNRJGEYJCDQK,"SPRINT Running Shoes For Men (Blue, Grey)",Vector,490.0,1.0,3.7,False
3,0.858,SHOFN3EJZVGC6GH8,RS-7250 Running Shoes For Women (Navy),Vector,990.0,9.0,3.9,False
4,0.855,SHOFN3EJVGC9VXYT,RS-7200 Running Shoes For Men (Navy),Vector,990.0,9.0,3.9,False
5,0.854,SHOEKNRJ34UCMAPA,"SPRINT Running Shoes For Men (Black, Grey)",Vector,490.0,1.0,3.7,False
6,0.854,SHOFN3EFNYGBKZT7,"RS-7050 Running Shoes For Men (Blue, Navy)",Vector,810.0,9.0,3.9,False
7,0.854,SHOESS9ZEZ6TEQUJ,Running Shoes For Men (Multicolor),Vector,1440.0,9.0,4.3,False
8,0.854,SHOESS9ZGMH7CURM,Running Shoes For Men (Multicolor),Vector,1440.0,9.0,3.7,False
9,0.852,SHOEP5GUQYYFKYSN,Running Shoes For Women (White),Vector,990.0,9.0,4.1,False
10,0.837,SHOEWQKVSGFGXPQF,Running Shoes For Men (Blue),Vector,1520.0,1.0,3.3,False


Query 7: men formal trousers (word2vec + cosine, top 20)


,score,pid,title,brand,selling_price,discount,average_rating,out_of_stock
1,0.817,TROFW4RYACEFP4SC,Slim Fit Men Beige Viscose Blend Trousers,R,449.0,65.0,4.0,False
2,0.815,TROFW4RYVXMERBZP,Slim Fit Men Black Viscose Blend Trousers,R,449.0,65.0,4.0,False
3,0.813,TROFW4RYKWKUJHTY,Slim Fit Men Grey Viscose Blend Trousers,R,449.0,65.0,4.0,False
4,0.812,TROEQQ7DZJ7FBGCC,Slim Fit Men Blue polyviscos Trousers,R,499.0,66.0,3.5,False
5,0.810,TROEQQ7DCCUGHZDE,Slim Fit Men Black polyviscos Trousers,R,549.0,63.0,3.5,False
6,0.805,TROFGZUTYEYRZJCC,Regular Fit Men Black Viscose Trousers,PIX,459.0,64.0,2.9,False
7,0.802,TROFW4RYFWJFPH4C,Slim Fit Men Gold Viscose Blend Trousers,R,449.0,65.0,4.0,False
8,0.789,TROFM47DXRTJWXYC,Slim Fit Men Beige Cotton Blend Trousers,True Bl,999.0,60.0,3.6,False
9,0.788,TROFNNPGKUCRM6MS,Slim Fit Men Grey Rayon Trousers,True Bl,919.0,60.0,3.6,False
10,0.788,TROFM47FSDAMFGQU,Slim Fit Men Blue Cotton Blend Trousers,True Bl,1039.0,60.0,3.6,False


In [22]:
# word2vec compared with the other methods on our 5 queries (top 10)
display(top_k_quality(RANKERS["word2vec"], TEST_QUERIES))
tables = {name: top_k_quality(ranker, TEST_QUERIES) for name, ranker in RANKERS.items()}
pd.DataFrame({name: {"distinct titles (avg)": t["distinct titles"].mean(),
                     "out of stock (total)": t["out of stock"].sum(),
                     "avg rating": t["avg rating"].mean(),
                     "wrong gender (total)": t["wrong gender"].sum(),
                     "avg length (terms)": t["avg length"].mean(),
                     "same as TF-IDF (avg)": t["same as TF-IDF"].mean()}
              for name, t in tables.items()}).round(2)

,distinct titles,out of stock,avg rating,wrong gender,avg length,same as TF-IDF
query,,,,,,
3,8,0,4.12,0,33.1,4
4,6,1,3.79,0,29.2,0
5,6,4,3.84,0,46.2,1
6,9,0,3.84,2,15.5,7
7,10,0,3.67,0,23.4,2


,TF-IDF,BM25,Our score,word2vec
distinct titles (avg),5.40,7.80,6.60,7.80
out of stock (total),2.00,4.00,0.00,5.00
avg rating,3.66,3.59,4.06,3.85
wrong gender (total),2.00,2.00,0.00,2.00
avg length (terms),24.44,30.04,29.06,29.48
same as TF-IDF (avg),10.00,3.40,3.80,2.80


In [23]:
w2v_table, w2v_summary = evaluate(RANKERS["word2vec"], TEST_QUERIES, ours)
display(w2v_table)
print({key: round(value, 3) for key, value in w2v_summary.items()})

,P@10,R@10,AP@10,F1@10,RR@10,NDCG@10,judged in top 10
Query 3,0.9,0.500,0.900,0.643,1.0,0.936,9.0
Query 4,1.0,0.500,1.000,0.667,1.0,1.000,2.0
Query 5,1.0,0.714,1.000,0.833,1.0,1.000,8.0
Query 6,0.8,0.667,0.692,0.727,1.0,0.824,10.0
Query 7,1.0,0.500,1.000,0.667,1.0,1.000,4.0


{'MAP': 0.918, 'MRR': 1.0}


## 3. A better representation than word2vec

Discussed in the report: averaging word vectors loses word order and lets frequent words dominate. Doc2vec and
sentence embeddings (e.g. Sentence-BERT) learn one vector for the whole text instead.

## 4. Extra: labels for the new results

Our Part 2 labels only cover the top 20 of TF-IDF. The next cell writes `our_queries_labels_part3_template.csv`
with the products that BM25 or our score put in their top 10, or word2vec in its top 20, **and that have no label
yet**. Fill the `labels` column (same criteria as Part 2), save it as `our_queries_labels_part3.csv` and run the
notebook again: section 0 adds these labels to the Part 2 ones.

In [24]:
TEMPLATE_PATH = Path("our_queries_labels_part3_template.csv")
labelled = set(zip(ours["query_id"], ours["pid"]))
rows = []
for qid, q in TEST_QUERIES.items():
    pool = {pid for name in ["BM25", "Our score"] for pid, _ in RANKERS[name](q, K)}
    pool |= {pid for pid, _ in RANKERS["word2vec"](q, 20)}
    for pid in sorted(pool):
        if (qid, pid) not in labelled:
            product = corpus_by_pid.loc[pid]
            rows.append({"title": product["title"], "pid": pid, "query_id": qid, "labels": "", "query": q,
                         "brand": product["brand"], "sub_category": product["sub_category"],
                         "details": product["details_text"]})
pd.DataFrame(rows).to_csv(TEMPLATE_PATH, index=False)
print(f"{len(rows)} products without a label written to project_progress/part_3/{TEMPLATE_PATH}")
print(pd.DataFrame(rows).groupby("query_id").size().to_dict() if rows else {})

54 products without a label written to project_progress/part_3/our_queries_labels_part3_template.csv
{3: 6, 4: 27, 5: 3, 6: 5, 7: 13}
